In [ ]:
import numpy as np
from datasets import load_dataset
import torch
import time
import matplotlib.pyplot as plt
from datasets import load_dataset
import random
import os
from  pprint import pprint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install matplotlib

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
device

# Load dataset

This is a dataset designed to enhance the overall truthfulness of LLMs, without sacrificing immersion when roleplaying as a human.
For example, in normal AI assistant model, the model should not try to describe what the warmth of the sun feels like, but if the system prompt indicates it's a human, it should.
Mostly targets corporeal, spacial, temporal awareness, and common misconceptions.([Hugging Face][1])

**Fields / schema (per record):**

* `prompt` — the user question or instruction
* `chosen` — the preferred (truthful/factual) response
* `rejected` — the dispreferred (hallucinated/incorrect) response

**Typical use-cases:** DPO / preference-based fine-tuning, alignment experiments to reduce hallucinations, ranking/evaluation for truthfulness. ([Hugging Face][1])


[1]: https://huggingface.co/datasets/jondurbin/truthy-dpo-v0.1/blame/46b2b8ca5fe89ef9776bbb2673c934f101f801b8/README.md?utm_source=chatgpt.com "README.md · jondurbin/truthy-dpo-v0.1 at ..."


In [ ]:
dataset = load_dataset("jondurbin/truthy-dpo-v0.1")

In [ ]:
pprint(dataset)

In [ ]:
pprint(dataset['train'])

In [ ]:
pprint(dataset['train'][0])

In [ ]:
pprint(dataset['train'].column_names)

In [ ]:
train_data = dataset["train"]

prompts = train_data["prompt"]
chosen_answers = train_data["chosen"]
rejected_answers = train_data["rejected"]

In [ ]:
prompts

In [ ]:
chosen_answers

In [ ]:
rejected_answers

In [ ]:
def prepare_data(example):
    # Ensure we have str types
    prompt = example.get('prompt', '')
    chosen = example.get('chosen', '')
    rejected = example.get('rejected', '')
    return {
        'prompt': prompt,
        'chosen': chosen,
        'rejected': rejected
    }

In [ ]:
processed = dataset['train'].map(prepare_data)
processed = processed.remove_columns([c for c in processed.column_names if c not in ['prompt','chosen','rejected']])
processed = processed.filter(lambda x: x['prompt'] and x['chosen'] and x['rejected'])

print(f"Processed examples: {len(processed)}")

In [ ]:
processed

# Training with DPOTrainer

### Supervised Fine-Tuning (SFT) on the dataset where inputs are prompt and target is chosen.

In [ ]:
!pip install trl

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer

In [ ]:
# Choose a model small enough to run in colab env
BASE_MODEL = "facebook/opt-1.3b"

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)

In [ ]:
print(tokenizer.all_special_tokens)
print(tokenizer.all_special_ids)
print(f"BOS: {tokenizer.bos_token} → id {tokenizer.bos_token_id}")
print(f"EOS: {tokenizer.eos_token} → id {tokenizer.eos_token_id}")
print(f"PAD: {tokenizer.pad_token} → id {tokenizer.pad_token_id}")
print(f"UNK: {tokenizer.unk_token} → id {tokenizer.unk_token_id}")

In [ ]:
#  enfore special token order and usages
tokenizer.add_special_tokens({
    "bos_token": "<s>",
    "eos_token": "</s>",
    "pad_token": "<pad>",
    "unk_token": "<unk>",
})

In [ ]:
text = ["the brown fox jumps over the lazy dog.", "fox"]
tok = tokenizer(text,padding=True)
tok

In [ ]:
tokenizer.decode(tok['input_ids'])

In [ ]:
tokenizer.all_special_tokens

In [ ]:
tokenizer.all_special_ids

In [ ]:
print(tokenizer.all_special_tokens)
print(tokenizer.all_special_ids)
print(f"BOS: {tokenizer.bos_token} → id {tokenizer.bos_token_id}")
print(f"EOS: {tokenizer.eos_token} → id {tokenizer.eos_token_id}")
print(f"PAD: {tokenizer.pad_token} → id {tokenizer.pad_token_id}")
print(f"UNK: {tokenizer.unk_token} → id {tokenizer.unk_token_id}")

In [ ]:
# Prepare SFT-format dataset: concat prompt + chosen as input-target pairs
max_length = 512

In [ ]:
def sft_format(example):
    '''
    Supervised Fine-Tuning (SFT) on the dataset where inputs are prompt and target is chosen.
    '''
    instruction = example['prompt']
    response = example['chosen']

    input_txt = (
        f"### Instruction:\n{instruction}\n\n" # headers — gives the model a clear, learnable separator between prompt and response (this is the Alpaca format, widely used for instruction tuning)
        f"### Response:\n{response}"
    )
    return {"text": input_txt}

In [ ]:
sft_format(processed[0])

In [ ]:
sft_dataset = processed.map(sft_format)

In [ ]:
sft_dataset

In [ ]:
sft_dataset['text'][0]

Train SFT model

In [ ]:
# remove all cols except curated text(prompt + chosen)
sft_dataset = sft_dataset.remove_columns(["prompt", "chosen", "rejected"])

In [ ]:
sft_dataset

In [ ]:
!pip install -U bitsandbytes>=0.46.1 accelerate -q
!pip install peft -q

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import BitsAndBytesConfig

# Load quantized model
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="auto",
)

# Attach LoRA adapters
lora_config = LoraConfig(
    r=16,                    # rank — higher = more params, more memory
    lora_alpha=32,           # scaling factor, usually 2x r
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],  # which layers to apply LoRA to
)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

class TrainSFT():
    sft_trainer = None

    def get_model(self):
        if not isinstance(self.sft_trainer, SFTTrainer):
            sft_trainer = SFTTrainer(
                        model=model,
                        processing_class=tokenizer,
                        train_dataset=sft_dataset,
                        peft_config=lora_config,
                        args=SFTConfig(
                            output_dir="./sft_checkpoint",
                            dataset_text_field="text",
                            completion_only_loss=False,
                            max_length=max_length,
                            per_device_train_batch_size=2,
                            gradient_accumulation_steps=8,
                            num_train_epochs=5,
                            learning_rate=2e-5,
                            fp16=False,
                            bf16=True,
                        )
                    )
            return sft_trainer
        else:
            return sft_trainer


In [ ]:
sft_trainer = TrainSFT().get_model()

In [ ]:
# train sft model
start_time = time.perf_counter()
sft_trainer.train()
end_time = (time.perf_counter() - start_time) * 1000
print(f"Execution time: {end_time} ms")

<img src="SFT train loss.png" width=600 height=400></img>

In [ ]:
# Save final model
sft_trainer.save_model("./sft_checkpoint/final") # saves LoRA adapters
tokenizer.save_pretrained("./sft_checkpoint/final") # save tokenizer | without it we can't reload the model correctly later.

In [ ]:
log_history = sft_trainer.state.log_history
# Extract just the losses
losses = [entry["loss"] for entry in log_history if "loss" in entry]
steps  = [entry["step"] for entry in log_history if "loss" in entry]

plt.plot(steps, losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("SFT Training Loss")
plt.show()

# DPO

In [ ]:
def dpo_format(example):
    return {
        "prompt":   f"### Instruction:\n{example['prompt']}\n\n### Response:\n",  # same format as SFT, but without response
        "chosen":   example['chosen'],
        "rejected": example['rejected'],
    }

In [ ]:
dpo_dataset = processed.map(dpo_format)
dpo_dataset

In [ ]:
dpo_dataset[0]

In [ ]:
max_length = 512 # same as SFT

In [ ]:
from peft import PeftModel, LoraConfig, get_peft_model
from trl import DPOTrainer, DPOConfig
from transformers import BitsAndBytesConfig


class TrainDPO:
    dpo_trainer = None

    # Reload base model
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map="auto",
    )

    # Load SFT LoRA adapters
    sft_model = PeftModel.from_pretrained(
        base_model, "/content/drive/MyDrive/lab5/sft_checkpoint"
    )

    # Merge LoRA weights into base model and unload adapters
    sft_model = (
        sft_model.merge_and_unload()
    )  # ← returns a plain model with LoRA baked in

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        "/content/drive/MyDrive/lab5/sft_checkpoint"
    )

    # Define new DPO LoRA config
    dpo_lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "v_proj"],
    )

    def train_dpo(self):
        if not isinstance(self.dpo_trainer, DPOTrainer):
            # 6. Initialize DPOTrainer with merged model + new lora config
            self.dpo_trainer = DPOTrainer(
                model=self.sft_model,  # ← plain merged model, not PeftModel
                processing_class=self.tokenizer,
                train_dataset=dpo_dataset,
                peft_config=self.dpo_lora_config,  # ← DPOTrainer applies fresh LoRA
                args=DPOConfig(
                    output_dir="/content/drive/MyDrive/lab5/dpo_checkpoint",
                    max_length=max_length,
                    per_device_train_batch_size=2,
                    gradient_accumulation_steps=8,
                    num_train_epochs=4,
                    learning_rate=1e-5,
                    bf16=True,
                    fp16=False,
                ),
            )
            return self.dpo_trainer
        else:
            return self.dpo_trainer


In [ ]:
dpo_trainer = TrainDPO().train_dpo()

In [ ]:
# train sft model
start_time = time.perf_counter()
dpo_trainer.train()
end_time = (time.perf_counter() - start_time) * 1000
print(f"Execution time: {end_time} ms")

<img src="DPO train loss.png" width=600 height=400></img>

In [ ]:
log_history = dpo_trainer.state.log_history
# Extract just the losses
losses = [entry["loss"] for entry in log_history if "loss" in entry]
steps  = [entry["step"] for entry in log_history if "loss" in entry]

plt.plot(steps, losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("DPO Training Loss")
plt.show()

In [ ]:
# save DPO model
dpo_trainer.save_model("/content/drive/MyDrive/lab5/dpo_checkpoint_final")
dpo_trainer.processing_class.save_pretrained("/content/drive/MyDrive/lab5/dpo_checkpoint_final")

## Push model to Hugging Face Hub

Uploaded link: https://huggingface.co/Imtiaz1256/opt-1.3b-dpo-finetuned

In [ ]:
from huggingface_hub import login
login(token="")

In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer

repo_name = "Imtiaz1256/opt-1.3b-dpo-finetuned" 

# Load from saved checkpoint and push
model = PeftModel.from_pretrained(model, "./dpo_checkpoint_final")
tokenizer = AutoTokenizer.from_pretrained("./dpo_checkpoint_final")

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

## Evaluation: LLM-as-a-Judge with AlpacaEval

In [ ]:
# Load AlpacaEval and sample 15 prompts
data_url = "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/alpaca_eval.json"
all_eval = load_dataset("json", data_files=data_url)
print(all_eval)

In [ ]:
# Filter for helpful_base
helpful_base = all_eval['train'].filter(lambda x: x.get('dataset','') == 'helpful_base')

# Sample 15 random
sampled = helpful_base.shuffle(seed=42).select(range(15))
sampled[0]

In [ ]:
# Create a function to generate answers from both models
from transformers import pipeline, AutoTokenizer
BASE_MODEL = "facebook/opt-1.3b"
# Load tokenizers from their respective checkpoints
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
dpo_tokenizer = AutoTokenizer.from_pretrained("./dpo_checkpoint_final")


# Load base and dpo models
base_pipe = pipeline('text-generation', model=BASE_MODEL, tokenizer=base_tokenizer, device=0)
dpo_pipe = pipeline('text-generation', model='./dpo_checkpoint_final', tokenizer=dpo_tokenizer, device=device)


The Judge Prompt

In [ ]:
def gen_answer(pipe, instruction, max_new_tokens=256):
    # We follow the Alpaca format: the instruction is directly fed
    outputs = pipe(instruction, max_new_tokens=max_new_tokens, do_sample=False)
    return outputs[0]['generated_text']

In [ ]:
records = []
for i, example in enumerate(sampled):
  instruction = example['instruction'] if 'instruction' in example else ''
  base_ans = gen_answer(base_pipe, instruction) # get ans from base model
  dpo_ans = gen_answer(dpo_pipe, instruction) # get ans from trained dpo model
  records.append({
        'id': i,
        'instruction': instruction,
        'base': base_ans,
        'dpo': dpo_ans
    })

In [ ]:
records[0]

In [ ]:
# judge prompt
judge_template = (
    "You are a highly qualified and impartial judge evaluating two AI models. Your task is to\n"
    "determine which model provides a better, more accurate, and more helpful response to the\n"
    "user’s instruction.\n\n"
    "User Instruction: {instruction}\n\n"
    "Model A (Base Model): {base}\n"
    "Model B (DPO Model): {dpo}\n\n"
    "Evaluate both models. Output ONLY your final verdict as exactly one of the following\n"
    "options, with no extra text or explanation: \"Model A\", \"Model B\", or \"Tie\"."
)

prompt = judge_template.format(instruction=records[0]['instruction'], base=records[0]['base'], dpo=records[0]['dpo'])
print(prompt)

Call Judge LLM and collect verdicts

In [ ]:
!pip install -U transformers accelerate bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

judge_model = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(judge_model)

Jmodel = AutoModelForCausalLM.from_pretrained(
    judge_model,
    quantization_config=bnb_config,
    device_map="auto"
)

judge = pipeline(
    "text-generation",
    model=Jmodel,
    tokenizer=tokenizer
)

In [ ]:
import re

def call_local_judge(prompt):
    messages = [
        {"role": "system", "content": "You are a fair AI judge."},
        {"role": "user", "content": prompt}
    ]

    # Build prompt for chat models; fallback to plain prompt if unavailable
    try:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        text = prompt + "\n\nAssistant:"

    # Generate deterministically with a short output budget
    out = judge(
        text,
        max_new_tokens=12,
        temperature=0.0,
        do_sample=False
    )
    generated = out[0]["generated_text"]

    # Keep only the generated tail (in case the model returned prompt+gen)
    gen_tail = generated[len(text):] if generated.startswith(text) else generated
    gen_tail = gen_tail.strip()

    # direct exact-token match (case-insensitive)
    for token in ("Model A", "Model B", "Tie"):
        if re.search(r'\b' + re.escape(token) + r'\b', gen_tail, flags=re.IGNORECASE):
            return token

    # normalized 
    plain = re.sub(r'[^a-z0-9\s]', ' ', gen_tail.lower())
    if "model a" in plain:
        return "Model A"
    if "model b" in plain:
        return "Model B"
    if "tie" in plain or "equal" in plain or "draw" in plain:
        return "Tie"

    # short single-letter outputs like "A" / "B"
    m = re.match(r'^\s*([ab])\b', plain)
    if m:
        return "Model A" if m.group(1) == "a" else "Model B"

    #fallback 
    return "Tie"

In [ ]:
prompt = judge_template.format(
    instruction=records[0]['instruction'],
    base=records[0]['base'],
    dpo=records[0]['dpo']
)

verdict = call_local_judge(prompt)

print(verdict)

In [ ]:
def evaluate(records):
    res = {"id": [], "text": [], "winner": []}
    for i, rec in enumerate(records):
        id_, instruction, base, dpo = [i[-1] for i in rec.items()]
        prompt = judge_template.format(instruction=instruction, base=base, dpo=dpo)
        verdict = call_local_judge(prompt)

        res["id"].append(int(id_) + 1)
        res["text"].append(instruction)
        
        res["winner"].append(verdict)

    return res


In [ ]:
evl_data = evaluate(records)

In [ ]:
print(f"Sample id{'':<10}Instruction (Truncated){'':<15}Winner (Judge)")
for i in range(len(evl_data['id'])):
    id_, ins, jud = evl_data['id'][i], evl_data['text'][i], evl_data['winner'][i]
    ins = ins.split()
    ins = ins[0:5]
    ins = ' '.join(ins) + '...'
    print(f"{id_:<20}{ins:<40}{jud}")

<p> Sample result of judge model<br>Model A(Base model)<br>Model B(Trained DPO model)</p>
<img src="sample_result.png" width=600 height=400></img>

## Calculate Win Rate

In [ ]:
def calculate_win_rate(res, tar):
    total_model_b = np.array([1 if i in tar else 0 for i in res])
    total_model_tie = np.array([1 if i in "Tie" else 0 for i in res])
    total_model_b_sum = np.sum(total_model_b)
    total_model_tie_sum = np.sum(total_model_tie)
    total_valid_eval = len(res)  
    win_rate = (total_model_b_sum + (0.5 * total_model_tie_sum)) / total_valid_eval
    return win_rate

In [ ]:
wr = calculate_win_rate(evl_data['winner'], "Model B")
print(f"Win rate: {wr:0.2f}%")

<img src="win_rate.png" ></img>